Здесь представлен код который занимается переводом строк через грок для проверки, т.к. это бейслайн явыбрал модель которую можно теоретически запустить на ПК - llama-3.1-8b-instant. Плюс у грока у нее больше всего токенов.

P.S. чем больше я работаю в ноутбуках, тем меньше мне они нравятся, просмотр их в пайчарме на маке или в браузере не обходится без артефактов рендера и дребезжания ячеек. Я надеюсь это локальная проблема, а не повреждение файла.
На всякий случай перенес все в коллаб

In [ ]:
import pathlib
import sqlite3
import sys
from typing import List, Dict

PROJECT_ROOT = pathlib.Path(".").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from translator_code.config import load_config
from translator_code.translator_core import Translator
from translator_code.cache_utils import load_cache, save_cache, sha1
from checkpoint3.llm_tranlator.translator_code.db_worker import fetch_valid_rows_for_translation,ensure_translated_table
from checkpoint3.llm_tranlator.translator_code import Config
from translator_code.translation_helpers import (
    _print_source_stats,
    _process_rows_with_cache_and_batches,
    _handle_row_with_cache,
    _apply_cached_translation,
    _flush_batch,
)
import logging

logging.getLogger("httpx").setLevel(logging.WARNING)
logging.getLogger("groq").setLevel(logging.WARNING)

#нужно отредактировать .env и добавить GROQ_API_KEY = токен_для_грока
DB_DIR = pathlib.Path("./db")
DB_FILENAME = "parallel_pairs_stream.sqlite"
DB_PATH = DB_DIR / DB_FILENAME

# Ссылка на БД на Google Drive
GDRIVE_URL = "https://drive.google.com/file/d/1rhrsSFS-4KZbK0HYTkmerjXpEsisf7NC/view?usp=share_link"

TARGET_MODEL_NAME = "llama-3.1-8b-instant"

DB_SOURCE_LANG = "en"   # язык исходного текста в формате для БД
DB_TARGET_LANG = "ru"   # таргет

/Users/sergejpolunin/PycharmProjects/StellarisEDA/mod_translation/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/Users/sergejpolunin/PycharmProjects/StellarisEDA/mod_translation/venv/lib/python3.12/site-packages/torchmetrics/utilities/imports.py:23: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import DistributionNotFound, get_distribution


Перенес строки для проверки кач-ва перевода в бд SQLite, так их проще подключать (нет зависимости от питон модулей как в пикле) и хранить (не нужно переживать за поломку csv). Если код для загрузки не работает бд можно скачать с гугл диска https://drive.google.com/file/d/1Dzp9Byecgm7Ib730-4E0qGRJDxMNm6iX/view?usp=share_lin и положить в папку db.

In [ ]:
def ensure_gdown_installed():
    """Устанавливает gdown, если его нет, и возвращает модуль."""
    try:
        import gdown  # noqa: F401
    except ImportError:
        print("gdown не найден")
    finally:
        import gdown
    return gdown


def normalize_gdrive_url(url: str) -> str:
    """
    Принимает ссылку вида
      https://drive.google.com/file/d/<ID>/view?usp=...
    и возвращает
      https://drive.google.com/uc?id=<ID>
    чтобы gdown мог её скачать.
    """
    if "uc?id=" in url:
        return url
    import re
    m = re.search(r"/d/([^/]+)/", url)
    if not m:
        return url
    file_id = m.group(1)
    return f"https://drive.google.com/uc?id={file_id}"


def download_db_if_needed():
    """
    Проверяет наличие БД по пути DB_PATH.
    Если файла нет — создаёт ./db и качает файл с Google Drive.
    """
    if DB_PATH.exists():
        print(f"База уже существует: {DB_PATH}")
        return

    if not GDRIVE_URL:
        raise ValueError("Не задана GDRIVE_URL с ссылкой на БД в Google Drive.")

    DB_DIR.mkdir(parents=True, exist_ok=True)

    gdown = ensure_gdown_installed()
    url = normalize_gdrive_url(GDRIVE_URL)

    print(f"Скачиваю базу из Google Drive в {DB_PATH} ...")
    gdown.download(url, str(DB_PATH), quiet=False)

    if not DB_PATH.exists():
        raise RuntimeError("Не удалось скачать БД с Google Drive.")
    print("База успешно скачана.")

Дальше идет функция для перевода строк (внутренности скрыты в импортируемых модулях, в них работа с бд и обертка на апи). Кеш и батчи нужны для оптимизации запросов к апи - батчи чтобы оптимальнее использовать токены, кеш чтобы во время отладки/перезапусков данные не попавшие в бд не пропали.  
Переведенные строки сохраняются в отдельную табличку в базе (в рамках экспериментов в табличке сохраняется какая модель перевела строку).  
Тк наша задача оценить кач-во бейслайна то для перевода мы будем брать не все строки а только те у которых исходные теги корректны (нет человеческих ошибок).  

Перевод работает следующим образом:

1. Текст из БД проходит через `protect_tokens` → плейсхолдеры превращаются в `<PH id="k0" />` и т.п.
2. Список защищённых строк передаётся в  
   `Translator.translate_batch(segments, segment_ids=...)`.
3. `Translator`:
   * следит за rate limit-ами и таймаутами;
   * может менять апи ключи и модели (в текущем примере модели не меняет);
   * отправляет запрос в Groq;
   * парсит ответ (JSON-массив);  
   * при проблемах с ответом пробует переотправить запрос, либо переходит на отдельные запросы для каждой строки.
4. Для списка переведённых строк вызывается `restore_tokens` (функция восстановления токенов), после чего строки попадают в БД.

Для модельки используется следующий системный промпт:
`You are a professional game localizer.
Translate user-visible text from {src} to {tgt}.
Do NOT translate or alter content inside <PH id=.../> tags.
Preserve placeholders, variables, icons, color codes, and explicit newlines.
STRICT OUTPUT RULES:
Respond with a JSON array ONLY.
NO comments, NO explanations, NO trailing commas, NO code fences.
Output must contain ONLY the JSON array of translated strings.
Array MUST have exactly the same length and order as the input.
Each element must be a plain string value.
Do NOT add anything else under any circumstances.`


С запросом отправляется:
`Translate this JSON array. Respond with a JSON array ONLY, no commentary:
<JSON-массив строк>`

Для перевода одиночной строки используется промпт (без требования JSON разметки):
`You are a professional game localizer.
Translate user-visible text from {src} to {tgt}.
Do NOT translate or alter content inside <PH id=.../> tags.
Preserve placeholders, variables, icons, color codes, and explicit newlines.
Return ONLY the translated string without additional commentary.`


In [ ]:
def translate_database_with_tag_filter(
    cfg: Config,
    tr: Translator,
    cache: Dict[str, str],
):
    """
    Верхнеуровневая функция:
      - берёт отфильтрованные строки (src+ref уже проверены),
      - гоняет их через кэш/батчи,
      - сохраняет переводы в БД.
    """
    rows, src_stats = fetch_valid_rows_for_translation(cfg)

    if not rows:
        print(
            "Нет строк для перевода после валидации.\n"
            f"Всего кандидатов: {src_stats['total_raw']}, "
            f"валидных: {src_stats['valid']}."
        )
        return

    _print_source_stats(src_stats)

    conn = sqlite3.connect(cfg.db_path)
    conn.row_factory = sqlite3.Row
    ensure_translated_table(conn)

    try:
        processed, saved = _process_rows_with_cache_and_batches(
            cfg, tr, cache, rows, conn
        )
    finally:
        save_cache(cfg.cache_path, cache)
        conn.close()

    print(
        "Перевод БД завершён.\n"
        f"  к переводу (после валидации src+ref): {len(rows)}\n"
        f"  реально обработано: {processed}\n"
        f"  сохранено: {saved}\n"
    )

Собственно запуск процесса. Для проверки мы переводим с en на ru.

In [ ]:
download_db_if_needed()

# грузим конфиг из .env и подправляем под этот ноутбук
cfg = load_config()

cfg.db_path = str(DB_PATH)
cfg.db_source_lang = DB_SOURCE_LANG
cfg.db_target_lang = DB_TARGET_LANG
cfg.model = TARGET_MODEL_NAME
cfg.model_fallbacks = ['']

print("Текущий конфиг для перевода:")
print("  db_path      =", cfg.db_path)
print("  db_source_lang =", cfg.db_source_lang)
print("  db_target_lang =", cfg.db_target_lang)
print("  model        =", cfg.model)

# создаём переводчик и кэш
tr = Translator(cfg)
cache = load_cache(cfg.cache_path)

# запускаем перевод с фильтром по целостности тегов, когда получим желаемое кол-во скрипт можно стопнуть
translate_database_with_tag_filter(cfg, tr, cache)

База уже существует: db/parallel_pairs_stream.sqlite
Текущий конфиг для перевода:
  db_path      = db/parallel_pairs_stream.sqlite
  db_source_lang = en
  db_target_lang = ru
  model        = llama-3.1-8b-instant
Статистика по исходным строкам:
  всего кандидатов: 47452
  валидных (src+ref ок): 46744
  пустые исходники: 0
  без референса: 0
  битые теги в исходнике: 596
  битые теги в референсе: 112



Translate rows:   4%|▎         | 1744/46744 [17:06<10:59:03,  1.14row/s]

[Translator] Переключаюсь: key gsk_KAL2…/model llama-3.1-8b-instant -> key gsk_MCLU…/model llama-3.1-8b-instant


Translate rows:   9%|▉         | 4389/46744 [46:35<5:19:11,  2.21row/s] 

[DB] Уже обработано 4400 строк из 46744 (сохранено 4400)


...

Не удалось перевести батч после смены ключей/моделей и нескольких попыток. Последняя ошибка: NotFoundError("Error code: 404 - {'error': {'message': 'The model `` does not exist or you do not have access to it.', 'type': 'invalid_request_error', 'code': 'model_not_found'}}")
Не удалось перевести батч после смены ключей/моделей и нескольких попыток. Последняя ошибка: NotFoundError("Error code: 404 - {'error': {'message': 'The model `` does not exist or you do not have access to it.', 'type': 'invalid_request_error', 'code': 'model_not_found'}}")
Не удалось перевести батч после смены ключей/моделей и нескольких попыток. Последняя ошибка: NotFoundError("Error code: 404 - {'error': {'message': 'The model `` does not exist or you do not have access to it.', 'type': 'invalid_request_error', 'code': 'model_not_found'}}")
Не удалось перевести батч после смены ключей/моделей и нескольких попыток. Последняя ошибка: NotFoundError("Error code: 404 - {'error': {'message': 'The model `` does not exis

Translate rows:  16%|█▌        | 7593/46744 [2:15:01<3162:03:38, 290.76s/row]

Не удалось перевести батч после смены ключей/моделей и нескольких попыток. Последняя ошибка: NotFoundError("Error code: 404 - {'error': {'message': 'The model `` does not exist or you do not have access to it.', 'type': 'invalid_request_error', 'code': 'model_not_found'}}")
Не удалось перевести батч после смены ключей/моделей и нескольких попыток. Последняя ошибка: NotFoundError("Error code: 404 - {'error': {'message': 'The model `` does not exist or you do not have access to it.', 'type': 'invalid_request_error', 'code': 'model_not_found'}}")
Не удалось перевести батч после смены ключей/моделей и нескольких попыток. Последняя ошибка: NotFoundError("Error code: 404 - {'error': {'message': 'The model `` does not exist or you do not have access to it.', 'type': 'invalid_request_error', 'code': 'model_not_found'}}")
Не удалось перевести батч после смены ключей/моделей и нескольких попыток. Последняя ошибка: NotFoundError("Error code: 404 - {'error': {'message': 'The model `` does not exis

Translate rows:  16%|█▌        | 7593/46744 [2:16:01<11:41:22,  1.07s/row]   


KeyboardInterrupt: 

На ошибку в конце можно не обращать внимание и стопать код перевода в любой момент, он не будет по второму кругу гнать уже переведенные строки